# ComplianceGPT — Batch Run (Rev4 + Rev5)

This notebook:
- mounts Google Drive
- installs minimal dependencies
- imports existing `pipeline.py` / `generator.py` from Drive (no `%%writefile`)
- runs batch benchmarks for **Rev5** and **Rev4**
- runs a lightweight content evaluation (semantic similarity + exact match)


In [1]:
# [CELL 1] MOUNT DRIVE + ADD PATHS
# ==========================================================
# 0) MOUNT DRIVE & CONFIGURE PATHS
# ==========================================================
from google.colab import drive
import sys
import os

drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/answerer_v0/pipeline')
sys.path.append('/content/drive/MyDrive/answerer_v0/generator')
sys.path.append('/content/drive/MyDrive/answerer_v0/retriever_ablation_study/QUR_generator')

print("Drive mounted and custom paths appended.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and custom paths appended.


In [2]:
# [CELL 2] INSTALL DEPENDENCIES
# ==========================================================
# 1) INSTALL DEPENDENCIES
# ==========================================================
!pip -q install transformers accelerate bitsandbytes sentence-transformers faiss-cpu pyyaml tqdm

print("Dependencies installed.")

Dependencies installed.


In [3]:
# [CELL 3] IMPORT PIPELINE (Robust)
# ==========================================================
# 2) IMPORT MODULES
# ==========================================================
import importlib

# We try to import generator first to ensure dependencies are loaded
try:
    import generator
    importlib.reload(generator)
    print("generator module loaded.")
except ImportError as e:
    print(f"Warning: Could not import generator: {e}")

try:
    import pipeline
    importlib.reload(pipeline) # Force reload if edited files on Drive
    from pipeline import ComplianceGPTPipeline
    print("ComplianceGPTPipeline imported successfully.")
except ImportError as e:
    print(f"FATAL: Could not import pipeline: {e}")
    print("   Please check the Drive files for syntax errors (e.g. regex parenthesis).")

generator module loaded.
[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib
ComplianceGPTPipeline imported successfully.


In [4]:
# [CELL 4] CONFIGURATION
# ==========================================================
# 3) CONFIG
# ==========================================================
import json
import pandas as pd
from tqdm import tqdm

# --- Gold datasets ---
REV5_GOLD = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv"
REV4_GOLD = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv"

# --- Outputs ---
REV5_OUT = "batch_run_results_rev5.csv"
REV4_OUT = "batch_run_results_rev4.csv"

# --- Pipeline knobs ---
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
USE_QUR = True        # Set False if QUR module is missing/broken
TOP_K = 5
USE_GENERATOR = True  # True = Gen+Contract; False = Retrieval Ablation
ORG_PROFILE = {}      # Empty profile => PARAMS_REQUIRED behavior

# --- Clause-level CCS (source of truth) ---
REV5_CCS = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CCS = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"

# If you're running in this chat sandbox, you can use:
# REV5_CCS = "/mnt/data/NIST_SP-800-53_rev5_catalog.jsonl"
# REV4_CCS = "/mnt/data/NIST_SP-800-53_rev4_catalog.jsonl"

DOC_FILTER_MODE = "all"          # "all" (recommended) or "prefer_statement_only"
STRICT_CCS_ASSERT = True

print("Config loaded.")

Config loaded.


In [5]:
# [CELL 5] BATCH RUNNER
# ==========================================================
# 4) BATCH RUNNER FUNCTION
# ==========================================================
def _pick(row: pd.Series, candidates, default=""):
    for c in candidates:
        if c in row and pd.notna(row[c]):
            return row[c]
    return default

def run_batch(framework_version: str, gold_csv: str, out_csv: str):
    print(f"\n=== Running {framework_version.upper()} benchmark ===")
    if not os.path.exists(gold_csv):
        print(f"File not found: {gold_csv}")
        return pd.DataFrame()

    df = pd.read_csv(gold_csv, encoding="latin-1")
    print(f"Loaded {len(df)} rows from: {gold_csv}")

    # Initialize Pipeline
    ccs_path = REV5_CCS if framework_version == "rev5" else REV4_CCS
    pipeline = ComplianceGPTPipeline(
        framework_version=framework_version,
        model_id=MODEL_ID,
        use_qur=USE_QUR,
        doc_filter_mode=DOC_FILTER_MODE,
        ccs_path=ccs_path,
        strict_ccs_assert=STRICT_CCS_ASSERT,
    )

    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df)):
        qid = _pick(r, ["query_id", "id", "qid"])
        question = _pick(r, ["question", "query", "prompt"])

        # We pass the full gold row in case the pipeline uses the verifier internally
        gold_row = r.to_dict()

        try:
            out = pipeline.answer(
                str(question),
                org_profile=ORG_PROFILE,
                top_k=TOP_K,
                use_generator=USE_GENERATOR,
                gold_row=gold_row
            )
            c = out.get("contract", {}) or {}

            # Extract verifier errors if present
            v_errors = out.get("verifier_errors", [])

            rows.append({
                "query_id": qid,
                "question": question,
                "generated_answer": c.get("answer_text_with_citation", c.get("answer_text", "")),
                "primary_citation": c.get("primary_citation", ""),
                "status": c.get("status", ""),
                "odp_required": json.dumps(c.get("odp_required_list", []), ensure_ascii=False),
                "verifier_errors": json.dumps(v_errors, ensure_ascii=False),
                "gold_answer": _pick(r, ["gold_answer", "answer", "reference_answer"])
            })
        except Exception as e:
            rows.append({
                "query_id": qid,
                "question": question,
                "generated_answer": f"ERROR: {str(e)}",
                "status": "ERROR",
                "verifier_errors": "[]"
            })

    out_df = pd.DataFrame(rows)
    out_df.to_csv(out_csv, index=False)

    print(f"Saved: {out_csv}")
    print("Status counts:")
    print(out_df["status"].value_counts(dropna=False))
    return out_df

In [6]:
# [CELL 6] RUN REV5
# ==========================================================
# 5) RUN REV5
# ==========================================================
df_rev5 = run_batch("rev5", REV5_GOLD, REV5_OUT)
df_rev5.head(3)


=== Running REV5 benchmark ===
Loaded 100 rows from: /content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv
[System] Loading model: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

100%|██████████| 100/100 [21:50<00:00, 13.10s/it]

Saved: batch_run_results_rev5.csv
Status counts:
status
PARAMS_REQUIRED    78
OK                 22
Name: count, dtype: int64


,query_id,question,generated_answer,primary_citation,status,odp_required,verifier_errors,gold_answer
0,,When must account managers and the designated ...,ac-9.3 Notification of Account Changes Notify ...,AC-9(3),PARAMS_REQUIRED,"[""ac-09.03_odp.01"", ""ac-09.03_odp.02""]","[""Low Citation Recall""]","Notify account managers and {{ insert: param, ..."
1,,What must an organization enforce regarding lo...,Enforce approved authorizations for logical ac...,AC-3,OK,[],"[""Low Citation Recall""]",Enforce approved authorizations for logical ac...
2,,What principle should be employed to limit use...,ac-6 Least Privilege Employ the principle of l...,AC-6,OK,[],"[""Hallucinated Citations: {'AC-6'}"", ""Low Cita...","Employ the principle of least privilege, allow..."


In [7]:
# [CELL 7] RUN REV4
# ==========================================================
# 6) RUN REV4
# ==========================================================
df_rev4 = run_batch("rev4", REV4_GOLD, REV4_OUT)
df_rev4.head(3)


=== Running REV4 benchmark ===
Loaded 36 rows from: /content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv
[System] Loading model: Qwen/Qwen2.5-7B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[Pipeline] QUR initialized.
[Pipeline] Retriever loading: /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl
[Retriever] Loading catalog from: /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl
[Retriever] Successfully loaded 826 documents.
[Retriever] Building BM25 index over 826 docs...
[Retriever] Building Dense index (intfloat/e5-small-v2) over 826 docs...


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

[Retriever] Loading Cross-Encoder for reranking...


100%|██████████| 36/36 [06:57<00:00, 11.60s/it]

Saved: batch_run_results_rev4.csv
Status counts:
status
PARAMS_REQUIRED    26
OK                 10
Name: count, dtype: int64


,query_id,question,generated_answer,primary_citation,status,odp_required,verifier_errors,gold_answer
0,1,What does AC-2 require the organization to do ...,The organization: Identifies and selects the f...,AC-2,PARAMS_REQUIRED,"[""ac-2_prm_1"", ""ac-2_prm_2"", ""ac-2_prm_3"", ""ac...","[""Low Citation Recall""]",The organization manages information system ac...
1,2,What principle does AC-6 require the organizat...,The organization employs the principle of leas...,AC-6,OK,[],"[""Hallucinated Citations: {'AC-6'}"", ""Low Cita...",The organization employs the principle of leas...
2,3,What security awareness training must the orga...,The organization provides basic security aware...,AT-2,PARAMS_REQUIRED,"[""at-2_prm_1""]","[""Hallucinated Citations: {'AT-2'}"", ""Low Cita...",The organization provides basic security aware...


In [8]:
# [CELL 8] EVALUATION
# ==========================================================
# 7) CONTENT EVALUATION (Semantic + Exact Match)
# ==========================================================
import re
import numpy as np
from sentence_transformers import SentenceTransformer, util

# Citation stripper
_CITE_RE = re.compile(r"\[NIST\s+SP\s*800-53.*?\]", flags=re.IGNORECASE)

def _clean(s: str) -> str:
    s = "" if s is None else str(s)
    s = _CITE_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("Loading embedding model (intfloat/e5-small-v2)...")
embed_model = SentenceTransformer("intfloat/e5-small-v2")

def evaluate(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.empty:
        print(f"--- {name}: No data to evaluate ---")
        return df

    df = df.copy()
    gen = df["generated_answer"].astype(str).map(_clean).tolist()
    gold = df["gold_answer"].astype(str).map(_clean).tolist()

    # Exact match
    df["exact_match"] = [int(g.strip().lower() == r.strip().lower()) for g, r in zip(gen, gold)]

    # Semantic similarity
    gen_texts = ["passage: " + x for x in gen]
    gold_texts = ["passage: " + x for x in gold]

    gen_emb = embed_model.encode(gen_texts, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
    gold_emb = embed_model.encode(gold_texts, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)

    sims = util.pairwise_cos_sim(gen_emb, gold_emb).cpu().numpy().reshape(-1)
    # Pairwise diagonal only (row i vs row i)
    # The utilization above computes N x N, we only need diagonal if we pass single lists
    # Optimized way for parallel lists:
    sims = [float(util.cos_sim(g, r)[0][0]) for g, r in zip(gen_emb, gold_emb)]

    df["semantic_score"] = sims

    print(f"\n--- {name} Results ---")
    print(f"Avg semantic_score: {np.mean(sims):.4f}")
    print(f"Exact match rate:   {df['exact_match'].mean():.4f}")
    return df

eval_rev5 = evaluate(pd.read_csv(REV5_OUT), "Rev5")
eval_rev4 = evaluate(pd.read_csv(REV4_OUT), "Rev4")

eval_rev5.to_csv("eval_rev5.csv", index=False)
eval_rev4.to_csv("eval_rev4.csv", index=False)
print("\n Saved evaluation CSVs.")

Loading embedding model (intfloat/e5-small-v2)...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


--- Rev5 Results ---
Avg semantic_score: 0.9282
Exact match rate:   0.0000


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


--- Rev4 Results ---
Avg semantic_score: 0.9112
Exact match rate:   0.0000

 Saved evaluation CSVs.
